# Tutorial 5: End-to-End Workflow

## Overview

This notebook demonstrates the complete Fioracle pipeline from raw data to optimized portfolio.

### Complete Pipeline:

```
1. Data Loading (DataPipeline)
   ↓
2. Feature Engineering (engineer_features)
   ↓
3. Regime Identification (Jump Model + HMM)
   ↓
4. Regime Forecasting (XGBoost)
   ↓
5. Lambda Tuning (Cross-Validation)
   ↓
6. Portfolio Optimization (RA-FIAP + RA-FIPO)
   ↓
7. Performance Evaluation
```

### Learning Objectives

1. Run complete workflow with minimal code
2. Understand integration between components
3. Generate publication-ready results
4. Save models and outputs for production use

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Create output directory
OUTPUT_DIR = Path('../output/tutorial')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Setup complete")
print(f"  Output directory: {OUTPUT_DIR}")

## Configuration

In [ ]:
# Configuration
CONFIG = {
    'data': {
        'mode': 'basic',  # 'basic' or 'full'
        'start_date': '1985-01-01',
        'end_date': '2010-12-31',
    },
    'regimes': {
        'initial_lambda': 5.0,
        'tune_lambda': False,  # Set True for lambda optimization
    },
    'portfolio': {
        'gamma_risk': 10.0,
        'gamma_trade': 1.0,
        'transaction_cost': 0.0005,
        'max_weight': 0.40,
    },
    'evaluation': {
        'test_size': 0.2,
    },
}

print("Configuration:")
for section, params in CONFIG.items():
    print(f"\n{section.upper()}:")
    for key, value in params.items():
        print(f"  {key}: {value}")

## Step 1: Load Data

In [ ]:
from src.core.data import DataPipeline

print("Step 1: Loading data...")

pipeline = DataPipeline(mode=CONFIG['data']['mode'])
raw_data = pipeline.load(
    start_date=CONFIG['data']['start_date'],
    end_date=CONFIG['data']['end_date']
)

print(f"✓ Loaded: {raw_data.shape}")
print(f"  Period: {raw_data.index.min()} to {raw_data.index.max()}")
print(f"  Features: {len(raw_data.columns)}")

## Step 2: Engineer Features

In [ ]:
from src.core.features import engineer_features

print("Step 2: Engineering features...")

asset_features, macro_features = engineer_features(
    raw_data, 
    complexity=CONFIG['data']['mode']
)

# Construct returns
asset_returns = {}
if 'shiller_sp500' in raw_data.columns:
    asset_returns['SP500'] = raw_data['shiller_sp500'].pct_change()
if 'shiller_gs10' in raw_data.columns:
    asset_returns['BOND_10Y'] = raw_data['shiller_gs10'].pct_change()
    asset_returns['CORP_AAA'] = raw_data['shiller_gs10'].pct_change() * 1.15
    asset_returns['CORP_BAA'] = raw_data['shiller_gs10'].pct_change() * 1.25

print(f"✓ Features engineered")
print(f"  Assets: {list(asset_returns.keys())}")
print(f"  Asset features: {list(asset_features.values())[0].shape[1] if asset_features else 0}")
print(f"  Macro features: {macro_features.shape[1]}")

## Step 3: Identify Regimes

In [ ]:
from src.models.jump_model import fit_all_asset_regimes
from src.models.regime_clustering import fit_all_regime_layers

print("Step 3: Identifying regimes...")

# Asset regimes (Layer C)
asset_regimes_results = fit_all_asset_regimes(
    asset_features=asset_features,
    asset_returns=asset_returns,
    lambda_jump=CONFIG['regimes']['initial_lambda']
)
asset_regimes = {name: result['regimes'] for name, result in asset_regimes_results.items()}

# Macro/Volatility regimes (Layers A/B)
try:
    regime_layers = fit_all_regime_layers(
        macro_features=macro_features,
        yield_data=raw_data[['shiller_gs10']] if 'shiller_gs10' in raw_data.columns else None,
        n_macro_states=3,
        n_volatility_states=3
    )
    print("✓ All regime layers identified")
except Exception as e:
    print(f"⚠ Macro/volatility regimes failed: {e}")
    regime_layers = None

# Show regime statistics
print("\nRegime Statistics:")
for asset, result in asset_regimes_results.items():
    stats = result['regime_stats']
    print(f"\n{asset}:")
    print(f"  Bullish: {stats.loc[0, 'count']} days ({stats.loc[0, 'count']/stats['count'].sum()*100:.1f}%)")
    print(f"  Bearish: {stats.loc[1, 'count']} days ({stats.loc[1, 'count']/stats['count'].sum()*100:.1f}%)")

## Step 4: Train Regime Forecasters

In [ ]:
from src.models.xgboost_classifier import prepare_supervised_dataset, train_classifiers_for_all_assets

print("Step 4: Training XGBoost forecasters...")

# Prepare datasets
supervised_data = prepare_supervised_dataset(
    asset_features=asset_features,
    asset_regimes=asset_regimes,
    macro_features=macro_features
)

# Train classifiers
xgb_results = train_classifiers_for_all_assets(
    supervised_data=supervised_data,
    test_size=CONFIG['evaluation']['test_size']
)

print("\n✓ XGBoost training complete")
print("\nTest Performance:")
for asset, results in xgb_results.items():
    acc = results['test_metrics']['accuracy']
    print(f"  {asset}: {acc:.4f} accuracy")

## Step 5: Lambda Tuning (Optional)

In [ ]:
from src.evaluation.cross_validation import tune_lambda_for_all_assets

if CONFIG['regimes']['tune_lambda']:
    print("Step 5: Tuning lambda parameters...")
    
    optimal_lambdas = tune_lambda_for_all_assets(
        asset_features=asset_features,
        asset_returns=asset_returns,
        lambda_candidates=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0],
        validation_years=5
    )
    
    print("\n✓ Lambda tuning complete")
    for asset, result in optimal_lambdas.items():
        print(f"  {asset}: λ={result['optimal_lambda']:.1f}")
else:
    print("Step 5: Skipping lambda tuning (using default values)")
    optimal_lambdas = {asset: {'optimal_lambda': CONFIG['regimes']['initial_lambda']} 
                      for asset in asset_regimes.keys()}

## Step 6: Portfolio Optimization

In [ ]:
from src.models.ra_fiap import generate_optimization_inputs
from src.models.portfolio_optimization import backtest_ra_fipo_portfolio

print("Step 6: Running portfolio optimization...")

# Get test period data
test_predictions = {asset: results['predictions_test'] 
                   for asset, results in xgb_results.items()}
predictions_df = pd.DataFrame(test_predictions)

# Prepare returns and regimes DataFrames
returns_df = pd.DataFrame(asset_returns)
regimes_df = pd.DataFrame(asset_regimes)

# Generate optimization inputs for each test date
mu_sigma_inputs = {}
for date in predictions_df.index:
    forecasts = predictions_df.loc[date].to_dict()
    
    try:
        inputs = generate_optimization_inputs(
            date=date,
            regime_forecasts=forecasts,
            returns_df=returns_df,
            regimes_df=regimes_df,
            optimal_lambdas={asset: optimal_lambdas[asset]['optimal_lambda'] 
                           for asset in forecasts.keys()},
            gpr_data=raw_data['macro_gpr'] if 'macro_gpr' in raw_data.columns else None
        )
        mu_sigma_inputs[date] = inputs
    except:
        pass

print(f"  Generated inputs for {len(mu_sigma_inputs)} dates")

# Run backtest
if len(mu_sigma_inputs) > 10:
    backtest_results = backtest_ra_fipo_portfolio(
        mu_sigma_inputs=mu_sigma_inputs,
        asset_returns=returns_df.loc[predictions_df.index],
        initial_weights=None,
        gamma_risk=CONFIG['portfolio']['gamma_risk'],
        gamma_trade=CONFIG['portfolio']['gamma_trade'],
        transaction_cost=CONFIG['portfolio']['transaction_cost'],
        max_weight=CONFIG['portfolio']['max_weight']
    )
    
    print("✓ Portfolio backtest complete")
else:
    print("⚠ Insufficient data for backtest")
    backtest_results = None

## Step 7: Performance Evaluation

In [ ]:
from src.evaluation.metrics import evaluate_strategy_performance

print("Step 7: Evaluating performance...")

# Evaluate strategy performance for each asset
performance_results = evaluate_strategy_performance(
    asset_returns={asset: returns_df[asset] for asset in asset_returns.keys()},
    predictions_df=pd.DataFrame({asset: xgb_results[asset]['predictions_test'] 
                                 for asset in asset_returns.keys()}),
    asset_regimes_df=regimes_df
)

print("\n✓ Performance evaluation complete")
print("\nStrategy Performance:")
print(performance_results[['asset', 'strategy_sharpe', 'bh_sharpe', 'sharpe_improvement']].to_string(index=False))

## Step 8: Visualizations

In [ ]:
# Portfolio cumulative returns
if backtest_results is not None:
    portfolio_returns = backtest_results['returns']['portfolio_return']
    portfolio_cum = (1 + portfolio_returns).cumprod()
    
    # Buy & hold comparison
    bh_returns = returns_df.loc[portfolio_returns.index].mean(axis=1)
    bh_cum = (1 + bh_returns).cumprod()
    
    # Plot
    fig, axes = plt.subplots(3, 1, figsize=(16, 14))
    
    # 1. Cumulative returns
    ax = axes[0]
    portfolio_cum.plot(ax=ax, linewidth=2, color='darkgreen', label='RA-FIPO Portfolio')
    bh_cum.plot(ax=ax, linewidth=2, color='steelblue', label='Buy & Hold', alpha=0.7)
    ax.set_title('Cumulative Returns', fontweight='bold', fontsize=14)
    ax.set_ylabel('Cumulative Return')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # 2. Drawdowns
    ax = axes[1]
    portfolio_dd = (portfolio_cum / portfolio_cum.cummax() - 1)
    bh_dd = (bh_cum / bh_cum.cummax() - 1)
    portfolio_dd.plot(ax=ax, linewidth=2, color='darkred', label='RA-FIPO')
    bh_dd.plot(ax=ax, linewidth=2, color='orange', label='Buy & Hold', alpha=0.7)
    ax.set_title('Drawdowns', fontweight='bold', fontsize=14)
    ax.set_ylabel('Drawdown (%)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}'))
    ax.legend()
    ax.grid(alpha=0.3)
    
    # 3. Portfolio weights
    ax = axes[2]
    backtest_results['weights'].plot(ax=ax, linewidth=1.5)
    ax.set_title('Portfolio Weights Evolution', fontweight='bold', fontsize=14)
    ax.set_ylabel('Weight (%)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.0f}'))
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'end_to_end_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Saved figure: {OUTPUT_DIR / 'end_to_end_results.png'}")

## Step 9: Save Results

In [ ]:
import pickle
import json

print("Step 9: Saving results...")

# Save performance summary
performance_results.to_csv(OUTPUT_DIR / 'performance_summary.csv', index=False)
print(f"  ✓ Saved: performance_summary.csv")

# Save optimal lambdas
with open(OUTPUT_DIR / 'optimal_lambdas.json', 'w') as f:
    json.dump(optimal_lambdas, f, indent=2)
print(f"  ✓ Saved: optimal_lambdas.json")

# Save XGBoost models
models_dir = OUTPUT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
for asset, results in xgb_results.items():
    model_file = models_dir / f'{asset}_xgboost.pkl'
    with open(model_file, 'wb') as f:
        pickle.dump(results['model'], f)
print(f"  ✓ Saved: {len(xgb_results)} XGBoost models")

# Save portfolio results
if backtest_results is not None:
    backtest_results['weights'].to_csv(OUTPUT_DIR / 'portfolio_weights.csv')
    backtest_results['returns'].to_csv(OUTPUT_DIR / 'portfolio_returns.csv')
    print(f"  ✓ Saved: portfolio_weights.csv, portfolio_returns.csv")

print(f"\n✅ All results saved to: {OUTPUT_DIR}")

## Final Summary

In [ ]:
print("="*80)
print("FIORACLE END-TO-END PIPELINE SUMMARY")
print("="*80)

print(f"\n📊 DATA:")
print(f"  Period: {CONFIG['data']['start_date']} to {CONFIG['data']['end_date']}")
print(f"  Mode: {CONFIG['data']['mode']}")
print(f"  Assets: {list(asset_returns.keys())}")

print(f"\n🎯 REGIMES:")
print(f"  Lambda: {CONFIG['regimes']['initial_lambda']}")
print(f"  Tuned: {'Yes' if CONFIG['regimes']['tune_lambda'] else 'No'}")

print(f"\n📈 FORECASTING:")
print(f"  Test size: {CONFIG['evaluation']['test_size']*100:.0f}%")
print(f"  Mean accuracy: {np.mean([r['test_metrics']['accuracy'] for r in xgb_results.values()]):.4f}")

print(f"\n💼 PORTFOLIO:")
if backtest_results is not None:
    portfolio_returns_series = backtest_results['returns']['portfolio_return']
    ann_return = portfolio_returns_series.mean() * 252 * 100
    ann_vol = portfolio_returns_series.std() * np.sqrt(252) * 100
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    cum_ret = (1 + portfolio_returns_series).cumprod()
    max_dd = (cum_ret / cum_ret.cummax() - 1).min() * 100
    
    print(f"  Annual Return: {ann_return:.2f}%")
    print(f"  Annual Volatility: {ann_vol:.2f}%")
    print(f"  Sharpe Ratio: {sharpe:.2f}")
    print(f"  Max Drawdown: {max_dd:.2f}%")
    
    # vs Buy & Hold
    bh_returns = returns_df.loc[portfolio_returns_series.index].mean(axis=1)
    bh_sharpe = (bh_returns.mean() / bh_returns.std() * np.sqrt(252)) if bh_returns.std() > 0 else 0
    print(f"\n  vs Buy & Hold:")
    print(f"    Sharpe improvement: +{sharpe - bh_sharpe:.2f}")

print(f"\n📁 OUTPUT:")
print(f"  Directory: {OUTPUT_DIR}")
print(f"  Files:")
for file in sorted(OUTPUT_DIR.glob('*')):
    print(f"    - {file.name}")

print(f"\n✅ Pipeline complete!")
print("="*80)

## Next Steps

### For Production Use:

1. **Load saved models**:
   ```python
   import pickle
   with open('output/tutorial/models/SP500_xgboost.pkl', 'rb') as f:
       model = pickle.load(f)
   ```

2. **Generate new predictions**:
   ```python
   new_features = compute_asset_features(new_returns)
   prediction = model.predict(new_features)
   ```

3. **Optimize portfolio**:
   ```python
   inputs = generate_optimization_inputs(date, forecasts, ...)
   weights = optimize_portfolio_weights(inputs['mu'], inputs['Sigma'], ...)
   ```

### For Further Exploration:

- **Tune hyperparameters**: Try different λ values, XGBoost params
- **Add assets**: Include more ETFs (TIP, HYG, etc.)
- **Extend period**: Use comprehensive mode for 1870-2025
- **Test robustness**: Cross-validation across multiple periods

### Resources:

- **Tutorials 1-4**: Deep dives into each component
- **Documentation**: See `docs/README.md`
- **Theory**: `JM_XGB_ENHANCED_SUMMARY.md`, `RA_FIPO_IMPLEMENTATION_SUMMARY.md`
- **Source code**: `src/core/` for streamlined modules

## 12. Bonus: Regime Drivers Visualization

Visualize how geopolitical and macroeconomic features drive asset regime predictions.

In [ ]:
from src.visualizations import visualize_regime_drivers

# Create comprehensive regime drivers visualization
fig = visualize_regime_drivers(
    start_date='2001-01-01',
    end_date='2010-12-31',
    output_path='../output/figures/regime_drivers_visualization.png',
    show_plot=True
)

print("\n✓ Visualization shows:")
print("  - Top row: GPR, EPU, VIX (macro drivers)")
print("  - Middle: 3-State HMM regime evolution")
print("  - Lower: Asset regime distribution & timeline")
print("  - See tutorial 06 for detailed explanation!")